In [ ]:
# Define the estimated policy as a dictionary with keys for all grid positions and corresponding actions.
estimated_policy = {
    (0, 0): 0, (0, 1): 1, (0, 2): 2, (0, 3): 3, (0, 4): 4,
    (0, 5): 5, (0, 6): 6, (1, 0): 0, (1, 1): 1, (1, 2): 2,
    (1, 3): 3, (1, 4): 4, (1, 5): 5, (1, 6): 6, (2, 0): 0,
    (2, 1): 1, (2, 2): 2, (2, 3): 3, (2, 4): 4, (2, 5): 5,
    (2, 6): 6, (3, 0): 0, (3, 1): 1, (3, 2): 2, (3, 3): 3,
    (3, 4): 4, (3, 5): 5, (3, 6): 6, (4, 0): 0, (4, 1): 1,
    (4, 2): 2, (4, 3): 3, (4, 4): 4, (4, 5): 5, (4, 6): 6,
    (5, 0): 0, (5, 1): 1, (5, 2): 2, (5, 3): 3, (5, 4): 4,
    (5, 5): 5, (5, 6): 6, (6, 0): 0, (6, 1): 1, (6, 2): 2,
    (6, 3): 3, (6, 4): 4, (6, 5): 5, (6, 6): 6
}


In [ ]:
class EaterEnvironment:
    def __init__(self, estimated_policy, grid_size=7, g1_state=37, g2_state=49, initial_bananas_g1=100, initial_bananas_g2=100):
        self.estimated_policy = estimated_policy
        self.grid_size = grid_size
        self.g1 = self.state_to_position(g1_state)
        self.g2 = self.state_to_position(g2_state)
        self.bananas_g1 = initial_bananas_g1
        self.bananas_g2 = initial_bananas_g2
        self.actions = [
            (1, 0), (0, 1), (0.5, 0.5), (0.2, 0.8), (0.6, 0.4),
            (0.8, 0.2), (0.4, 0.6), (0.7, 0.3), (0.3, 0.7)
        ]

    def state_to_position(self, state):
        x = (state - 1) % self.grid_size
        y = (state - 1) // self.grid_size
        return (x, y)

    def calculate_distance(self, pos1, pos2):
        return abs(pos1[0] - pos2[0]) + abs(pos1[1] - pos2[1])

    def calculate_reward(self, distance_to_g1, distance_to_g2, action_index):
      if distance_to_g1 + distance_to_g2 == 0:
          return 0  # Avoid division by zero, although unlikely in practical scenarios

      # Ratio of how much should ideally be consumed from g1 based on its relative closeness
      ideal_g1_ratio = distance_to_g2 / (distance_to_g1 + distance_to_g2)
      ideal_g2_ratio = 1 - ideal_g1_ratio  # Complement of g1 ratio for g2

      # Actions defined with their consumption ratios from g1 and g2
      action_ratios = {
          0: (1.0, 0.0),  # All from g1
          1: (0.0, 1.0),  # All from g2
          2: (0.5, 0.5),  # Equally from both
          3: (0.2, 0.8),  # Mainly from g2
          4: (0.6, 0.4),  # Mainly from g1
          5: (0.8, 0.2),  # Mainly from g1
          6: (0.4, 0.6),  # Mainly from g2
          7: (0.7, 0.3),  # Mainly from g1
          8: (0.3, 0.7)   # Mainly from g2
      }

      actual_g1_ratio, actual_g2_ratio = action_ratios[action_index]

      # Calculate the absolute differences between actual ratios and ideal ratios
      diff_g1 = abs(actual_g1_ratio - ideal_g1_ratio)
      diff_g2 = abs(actual_g2_ratio - ideal_g2_ratio)

      # Evaluate the reward based on how well the actual ratios align with the ideal ratios
      reward = 1 - (diff_g1 + diff_g2)  # Reward is higher when the differences are smaller

      # Scale the reward to fit into specific reward levels you mentioned
      if reward > 0.95:
          return 1
      elif reward > 0.9:
          return 0.8
      elif reward > 0.85:
          return 0.7
      elif reward > 0.8:
          return 0.6
      elif reward > 0.75:
          return 0.4
      elif reward > 0.7:
          return 0.3
      elif reward > 0.65:
          return 0.2
      elif reward > 0.6:
          return 0
      else:
          return -1





    def step(self, eater_state, action_index):
        mover_current_state, _ = eater_state
        mover_position = self.state_to_position(mover_current_state)

        # Make sure that mover_position is correctly formatted as (x, y) or (row, column)
        mover_next_action = self.estimated_policy[mover_position]  # Use tuple for grid position as the key
        mover_next_position = self.update_position(mover_position, mover_next_action)
        next_mover_state = self.position_to_state(mover_next_position)

        distance_to_g1 = self.calculate_distance(mover_next_position, self.g1)
        distance_to_g2 = self.calculate_distance(mover_next_position, self.g2)
        goal1_consume, goal2_consume = self.actions[action_index]
        self.bananas_g1 -= goal1_consume
        self.bananas_g2 -= goal2_consume

        reward = self.calculate_reward(distance_to_g1, distance_to_g2, action_index)

        # Ensure the next action is determined from the estimated_policy if used elsewhere
        #next_mover_action = self.estimated_policy[next_mover_position] # Uncomment if needed

        next_state = (next_mover_state, mover_next_action)
        return next_state, reward


    def update_position(self, position, action):
        x, y = position
        if action == 0 and y < self.grid_size - 1:
            y += 1
        elif action == 1 and y > 0:
            y -= 1
        elif action == 2 and x > 0:
            x -= 1
        elif action == 3 and x < self.grid_size - 1:
            x += 1
        return (x, y)

    def position_to_state(self, position):
        x, y = position
        return y * self.grid_size + x + 1
    def reset(self):
              # Reset the banana counts and the mover's position to the initial settings
              self.bananas_g1 = 100
              self.bananas_g2 = 100
              self.current_mover_state = random.choice(range(1, 50))
              self.current_mover_action = random.choice(range(0, 4))
              return (self.current_mover_state, self.current_mover_action)


In [ ]:

import random

env = EaterEnvironment(estimated_policy)
next_state = (1,0)
for i in range(10):
  action = random.choice(range(len(env.actions)))
  next_state, reward = env.step(next_state, action)
  print(next_state,action,reward)


(8, 0) 1 -1
(1, 1) 0 -1
(8, 0) 0 -1
(1, 1) 5 0.3
(8, 0) 4 0.6
(1, 1) 4 0.7
(8, 0) 1 -1
(1, 1) 8 -1
(8, 0) 2 0
(1, 1) 8 -1


In [ ]:
import numpy as np
import random

def initialize_q_returns_policy(env):
    Q = {}
    returns = {}
    policy = {}
    # Assuming there are 49 mover positions and 5 mover actions (0 to 4)
    possible_actions = range(len(env.actions))  # Assuming env.actions is defined with 9 possible eater actions

    # Initialize Q, returns, and policy for every possible state and action
    for mover_position in range(1, 50):  # Mover positions 1 to 49
        for mover_action in range(5):  # Mover actions 0 to 4
            state = (mover_position, mover_action)
            for eater_action in possible_actions:
                Q[(state, eater_action)] = 0  # Initial Q-value
                returns[(state, eater_action)] = []  # Initialize return list
            # Initialize policy with a random action from possible eater actions
            policy[state] = random.choice(list(possible_actions))

    return Q, returns, policy

def generate_episode(env, policy, epsilon=0.1):
    episode = []
    current_state = (1, 4)  # Adjust as needed
    done = False
    while not done:
        if random.random() < epsilon:
            action = random.choice(range(len(env.actions)))
        else:
            action = policy[current_state] if current_state in policy else random.choice(range(len(env.actions)))
        next_state, reward = env.step(current_state, action)
        episode.append((current_state, action, reward))
        if next_state == current_state:  # Assuming this is a termination condition
            done = True
        current_state = next_state
    return episode

def first_visit_mc(env, num_episodes, epsilon=0.1):
    Q, returns, policy = initialize_q_returns_policy(env)

    for episode_number in range(num_episodes):
        episode = generate_episode(env, policy, epsilon)
        G = 0
        visited = set()
        for (state, action, reward) in reversed(episode):
            G = reward + 0.99 * G
            if (state, action) not in visited:
                visited.add((state, action))
                returns[(state, action)].append(G)
                Q[(state, action)] = np.mean(returns[(state, action)])
                best_action = max((Q[(state, a)], a) for a in range(len(env.actions)))[1]
                policy[state] = best_action

    return policy, Q

estimated_policy = {
    (0, 0): 0, (0, 1): 0, (0, 2): 0, (0, 3): 0, (0, 4): 2,
    (0, 5): 2, (0, 6): 2, (1, 0): 0, (1, 1): 3, (1, 2): 3,
    (1, 3): 0,  (1, 4): 1, (1, 5): 1, (1, 6): 1, (2, 0): 0,
    (2, 1): 0, (2, 2): 0, (2, 3): 0, (2, 4): 0, (2, 5): 2,
    (2, 6): 2, (3, 0): 0, (3, 1): 3, (3, 2): 0, (3, 3): 0,
    (3, 4): 0, (3, 5): 0, (3, 6): 0, (4, 0): 0, (4, 1): 1,
    (4, 2): 0, (4, 3): 0, (4, 4): 2, (4, 5): 0, (4, 6): 2,
    (5, 0): 3, (5, 1): 3, (5, 2): 3, (5, 3): 3, (5, 4): 0,
    (5, 5): 1, (5, 6): 1, (6, 0): 0, (6, 1): 2, (6, 2): 2,
    (6, 3): 2, (6, 4): 2, (6, 5): 2, (6, 6): 3
}

env = EaterEnvironment(estimated_policy)
state = env.reset()
# Run the first-visit Monte Carlo
estimated_policy1, Q_values = first_visit_mc(env, 10000, 0.1)


In [ ]:
print(estimated_policy1)

{(1, 0): 6, (1, 1): 3, (1, 2): 5, (1, 3): 6, (1, 4): 7, (2, 0): 6, (2, 1): 3, (2, 2): 3, (2, 3): 1, (2, 4): 8, (3, 0): 6, (3, 1): 7, (3, 2): 6, (3, 3): 8, (3, 4): 8, (4, 0): 6, (4, 1): 8, (4, 2): 1, (4, 3): 6, (4, 4): 6, (5, 0): 2, (5, 1): 7, (5, 2): 6, (5, 3): 2, (5, 4): 5, (6, 0): 1, (6, 1): 0, (6, 2): 4, (6, 3): 7, (6, 4): 4, (7, 0): 1, (7, 1): 1, (7, 2): 6, (7, 3): 7, (7, 4): 3, (8, 0): 7, (8, 1): 8, (8, 2): 1, (8, 3): 7, (8, 4): 6, (9, 0): 1, (9, 1): 8, (9, 2): 4, (9, 3): 5, (9, 4): 7, (10, 0): 0, (10, 1): 1, (10, 2): 8, (10, 3): 3, (10, 4): 6, (11, 0): 2, (11, 1): 1, (11, 2): 6, (11, 3): 6, (11, 4): 1, (12, 0): 3, (12, 1): 1, (12, 2): 6, (12, 3): 6, (12, 4): 8, (13, 0): 2, (13, 1): 8, (13, 2): 0, (13, 3): 1, (13, 4): 7, (14, 0): 1, (14, 1): 4, (14, 2): 2, (14, 3): 1, (14, 4): 7, (15, 0): 5, (15, 1): 7, (15, 2): 7, (15, 3): 4, (15, 4): 4, (16, 0): 4, (16, 1): 2, (16, 2): 2, (16, 3): 4, (16, 4): 8, (17, 0): 7, (17, 1): 4, (17, 2): 5, (17, 3): 0, (17, 4): 0, (18, 0): 6, (18, 1): 5, 

In [ ]:
env = EaterEnvironment(estimated_policy)
current_eater_state = (1, 4)  # Initial state

for i in range(19):
    action_index = estimated_policy1[current_eater_state] # Cycle through actions
    next_state, reward = env.step(current_eater_state, action_index)
    print(f"Step {i+1}: Next State: {next_state}, Reward: {reward}, action: {env.actions[action_index]} Bananas at g1: {env.bananas_g1}, Bananas at g2: {env.bananas_g2}")
    current_eater_state = next_state

Step 1: Next State: (8, 0), Reward: 1, action: (0.7, 0.3) Bananas at g1: 99.3, Bananas at g2: 99.7
Step 2: Next State: (15, 0), Reward: 1, action: (0.7, 0.3) Bananas at g1: 98.6, Bananas at g2: 99.4
Step 3: Next State: (22, 0), Reward: 0.7, action: (0.8, 0.2) Bananas at g1: 97.8, Bananas at g2: 99.2
Step 4: Next State: (29, 0), Reward: 1, action: (0.8, 0.2) Bananas at g1: 97.0, Bananas at g2: 99.0
Step 5: Next State: (29, 2), Reward: 1, action: (0.8, 0.2) Bananas at g1: 96.2, Bananas at g2: 98.8
Step 6: Next State: (29, 2), Reward: 1, action: (0.8, 0.2) Bananas at g1: 95.4, Bananas at g2: 98.6
Step 7: Next State: (29, 2), Reward: 1, action: (0.8, 0.2) Bananas at g1: 94.60000000000001, Bananas at g2: 98.39999999999999
Step 8: Next State: (29, 2), Reward: 1, action: (0.8, 0.2) Bananas at g1: 93.80000000000001, Bananas at g2: 98.19999999999999
Step 9: Next State: (29, 2), Reward: 1, action: (0.8, 0.2) Bananas at g1: 93.00000000000001, Bananas at g2: 97.99999999999999
Step 10: Next State: 